In [1]:
import pandas as pd
from pathlib import Path

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

print("Raw path:", raw_path.resolve())
print("Processed path:", processed_path.resolve())

Raw path: D:\bluestock-mf\data\raw
Processed path: D:\bluestock-mf\data\processed


In [2]:
fund_master = pd.read_csv(raw_path / "01_fund_master.csv")
nav_history = pd.read_csv(raw_path / "02_nav_history.csv")
aum = pd.read_csv(raw_path / "03_aum_by_fund_house.csv")
sip = pd.read_csv(raw_path / "04_monthly_sip_inflows.csv")
category_inflows = pd.read_csv(raw_path / "05_category_inflows.csv")
folio_count = pd.read_csv(raw_path / "06_industry_folio_count.csv")
scheme_performance = pd.read_csv(raw_path / "07_scheme_performance.csv")
transactions = pd.read_csv(raw_path / "08_investor_transactions.csv")
holdings = pd.read_csv(raw_path / "09_portfolio_holdings.csv")
benchmark = pd.read_csv(raw_path / "10_benchmark_indices.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
datasets = {
    "01_fund_master": fund_master,
    "02_nav_history": nav_history,
    "03_aum_by_fund_house": aum,
    "04_monthly_sip_inflows": sip,
    "05_category_inflows": category_inflows,
    "06_industry_folio_count": folio_count,
    "07_scheme_performance": scheme_performance,
    "08_investor_transactions": transactions,
    "09_portfolio_holdings": holdings,
    "10_benchmark_indices": benchmark
}

In [4]:
overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

overview_df = pd.DataFrame(overview)

print(overview_df.to_string(index=False))

                 dataset  rows  columns  missing_values  duplicate_rows
          01_fund_master    40       15               0               0
          02_nav_history 46000        3               0               0
    03_aum_by_fund_house    90        5               0               0
  04_monthly_sip_inflows    48        6              12               0
     05_category_inflows   144        3               0               0
 06_industry_folio_count    21        6               0               0
   07_scheme_performance    40       19               0               0
08_investor_transactions 32778       13               0               0
   09_portfolio_holdings   322        8               0               0
    10_benchmark_indices  8050        3               0               0


In [6]:
print("Shape:", fund_master.shape)
print("Columns:")

print(fund_master.columns.tolist())
print("Fund houses:")

print(fund_master["fund_house"].value_counts())
print("Categories:")

print(fund_master["category"].value_counts())
print("Plans:")
print(fund_master["plan"].value_counts())

Shape: (40, 15)
Columns:
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
Fund houses:
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3
Name: count, dtype: int64
Categories:
category
Equity    34
Debt       6
Name: count, dtype: int64
Plans:
plan
Regular    32
Direct      8
Name: count, dtype: int64


In [7]:
print("Duplicate AMFI codes:", fund_master["amfi_code"].duplicated().sum())

Duplicate AMFI codes: 0


In [ ]:
print("Missing AMFI codes:", fund_master["amfi_code"].isna().sum())

np.int64(0)

In [10]:
print("Unique AMFI codes:", fund_master["amfi_code"].nunique())

Unique AMFI codes: 40


In [11]:
print("Unique fund houses:", fund_master["fund_house"].nunique())

Unique fund houses: 10


In [14]:
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [15]:
master_codes = set(fund_master["amfi_code"].astype(int))
nav_codes = set(nav_history["amfi_code"].astype(int))

missing_from_nav = sorted(master_codes - nav_codes)
extra_in_nav = sorted(nav_codes - master_codes)

print("Fund master codes:", len(master_codes))
print("NAV codes:", len(nav_codes))
print("Missing from NAV:", missing_from_nav)
print("Extra in NAV:", extra_in_nav)
print("Validation passed:", len(missing_from_nav) == 0 and len(extra_in_nav) == 0)

Fund master codes: 40
NAV codes: 40
Missing from NAV: []
Extra in NAV: []
Validation passed: True


In [16]:
nav_per_fund = nav_history.groupby("amfi_code").size()

print("Minimum records per fund:", nav_per_fund.min())
print("Maximum records per fund:", nav_per_fund.max())
print("Average records per fund:", nav_per_fund.mean())
print("Duplicate rows:", nav_history.duplicated().sum())
print("Duplicate fund-date combinations:", nav_history.duplicated(["amfi_code", "date"]).sum())
print("Missing NAV:", nav_history["nav"].isna().sum())
print("NAV <= 0:", (nav_history["nav"] <= 0).sum())

Minimum records per fund: 1150
Maximum records per fund: 1150
Average records per fund: 1150.0
Duplicate rows: 0
Duplicate fund-date combinations: 0
Missing NAV: 0
NAV <= 0: 0


In [17]:
aum.head()

,date,fund_house,fund_house_aum_lakh_crore,fund_house_aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [18]:
aum["date"].nunique()

9

In [19]:
aum["date"].min(), "to", aum["date"].max()

('2022-03-31', 'to', '2025-12-31')

In [20]:
print("Shape:", sip.shape)
print(sip.head())
print("\nDate range:", sip["month"].min(), "to", sip["month"].max())
print("Missing values:")
print(sip.isna().sum())

Shape: (48, 6)
     month  sip_inflow_crore  active_sip_accounts_crore  \
0  2022-01             11517                       4.91   
1  2022-02             11438                       4.93   
2  2022-03             12328                       5.09   
3  2022-04             11863                       5.48   
4  2022-05             12286                       5.55   

   new_sip_accounts_lakh  sip_aum_lakh_crore  yoy_growth_pct  
0                   9.10                4.80             NaN  
1                   8.20                4.85             NaN  
2                  10.50                5.01             NaN  
3                   9.52                5.12             NaN  
4                   8.10                5.15             NaN  

Date range: 2022-01 to 2025-12
Missing values:
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

In [21]:
print("Shape:", category_inflows.shape)
print(category_inflows.head())
print("\nCategories:", category_inflows["category"].nunique())
print("Months:", category_inflows["month"].nunique())
print("Date range:", category_inflows["month"].min(), "to", category_inflows["month"].max())

Shape: (144, 3)
     month         category  net_inflow_crore
0  2024-04        Large Cap            2413.0
1  2024-04          Mid Cap            3897.0
2  2024-04        Small Cap            3533.0
3  2024-04        Flexi Cap            4947.0
4  2024-04  Large & Mid Cap            4214.0

Categories: 12
Months: 12
Date range: 2024-04 to 2025-03


In [22]:
print("Shape:", transactions.shape)
print(transactions.head())
print("\nTransaction types:")
print(transactions["transaction_type"].value_counts())
print("\nKYC:")
print(transactions["kyc_status"].value_counts())
print("\nUnique investors:", transactions["investor_id"].nunique())

Shape: (32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   

In [23]:
print("Shape:", scheme_performance.shape)
print(scheme_performance.head())
print("\nUnique AMFI codes:", scheme_performance["amfi_code"].nunique())
print("\nMissing values:")
print(scheme_performance.isna().sum())

Shape: (40, 19)
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0   

In [24]:
print("Holdings:", holdings.shape)
print("Benchmark:", benchmark.shape)

print("\nBenchmark indices:")
print(benchmark["index_name"].value_counts())

print("\nHolding sectors:")
print(holdings["sector"].value_counts())

Holdings: (322, 8)
Benchmark: (8050, 3)

Benchmark indices:
index_name
NIFTY50            1150
NIFTY100           1150
NIFTY_MIDCAP150    1150
BSE_SMALLCAP       1150
NIFTY500           1150
CRISIL_LIQUID      1150
CRISIL_GILT        1150
Name: count, dtype: int64

Holding sectors:
sector
Banking           60
IT                40
Pharma            38
Automobile        33
Utilities         24
Infrastructure    22
FMCG              21
Telecom           15
Diversified       14
Energy            13
Cement            12
NBFC              11
Paints            10
Consumer Goods     9
Name: count, dtype: int64
